In [0]:
# 1. Read from the Bronze table
bronze_df = spark.read.table("yes_catalog.bronze.stations_raw")

# 2. Clean and Flatten
from pyspark.sql.functions import col

silver_df = bronze_df.select(
    col("stationcode").alias("station_id"),
    col("name"),
    col("capacity").cast("int"),
    col("numbikesavailable").cast("int").alias("bikes_available"),
    col("numdocksavailable").cast("int").alias("docks_available"),
    # Flatten the nested coordinates
    col("coordonnees_geo.lat").alias("latitude"),
    col("coordonnees_geo.lon").alias("longitude"),
    col("ingestion_time")
).filter("station_id IS NOT NULL AND capacity > 0")

# 3. Save to Silver (Overwrite is fine here for a lab)
silver_df.write.format("delta").mode("overwrite").saveAsTable("yes_catalog.bronze.stations_silver")

display(silver_df)

station_id,name,capacity,bikes_available,docks_available,latitude,longitude,ingestion_time
40001,Hôpital Mondor,28,6,22,48.798922410229,2.4537451531298,2026-04-29T13:03:18.010954Z
32304,Charcot - Benfleet,28,1,26,48.878370277021,2.440523876268,2026-04-29T13:03:18.010954Z
14111,Cassini - Denfert-Rochereau,25,5,19,48.837525839067,2.3360354080796,2026-04-29T13:03:18.010954Z
11104,Charonne - Robert et Sonia Delaunay,20,2,18,48.855907555969,2.3925706744194,2026-04-29T13:03:18.010954Z
7002,Vaneau - Sèvres,35,19,14,48.848563233059,2.3204218259346,2026-04-29T13:03:18.010954Z
5110,Lacépède - Monge,23,5,18,48.84389286531899,2.3519663885235786,2026-04-29T13:03:18.010954Z
6108,Saint-Romain - Cherche-Midi,17,6,9,48.84708159081946,2.321374788880348,2026-04-29T13:03:18.010954Z
33006,André Karman - République,31,10,21,48.91039875761846,2.3851355910301213,2026-04-29T13:03:18.010954Z
42016,Pierre et Marie Curie - Maurice Thorez,27,3,23,48.81580226360801,2.376804985105991,2026-04-29T13:03:18.010954Z
30002,Jean Rostand - Paul Vaillant Couturier,40,8,32,48.908168131015,2.4530601033354,2026-04-29T13:03:18.010954Z


In [0]:
# 1. Create a business-ready summary
gold_df = spark.sql("""
    SELECT 
        name, 
        AVG(bikes_available) as avg_bikes,
        AVG(bikes_available / capacity) * 100 as availability_rate
    FROM yes_catalog.bronze.stations_silver
    GROUP BY name
    ORDER BY availability_rate DESC
""")

# 2. Save to Gold
gold_df.write.format("delta").mode("overwrite").saveAsTable("yes_catalog.bronze.stations_gold")

display(gold_df)

name,avg_bikes,availability_rate
Erasme - Ulm,24.0,100.0
Henry Farman,16.0,100.0
Beaux-Arts - Bonaparte,20.0,100.0
Quai des Célestins - Henri IV,12.0,100.0
Quai Panhard et Levassor,55.0,96.49122807017544
Place de Barcelone - Mirabeau,51.0,92.72727272727272
Hôtel de Ville de Boulogne Billancourt,24.0,92.3076923076923
Desaix - Edgar Faure,11.0,91.66666666666666
Pesaro - Préfecture,20.0,90.9090909090909
Amelot - Saint-Sébastien,27.0,90.0
